In [ ]:
# Setup 1 — install a version-consistent Java / Spark / Deequ stack.
#
#   PyDeequ  → accepts SPARK_VERSION=3.5 only  (see pydeequ/configs.py)
#   Deequ    → the JAR it selects is a Scala 2.12 build
#   Spark    → 3.5.x runs on Java 8 / 11 / 17
#   PySpark  → 3.5.x supports Python 3.8–3.11
#
# Spark 4.x is the trap: it needs Java 17+ AND is Scala 2.13, so it cannot load
# the Scala 2.12 Deequ JAR that PyDeequ selects. Pin Spark 3.5.
#
# Two Colab-specific hazards are handled here:
#
#   * Downgrading over an existing PySpark 4.x leaves a MIXED tree — pip does
#     not remove every file, so you end up with 3.5's pyspark/pandas/internal.py
#     importing get_column_class from a 4.0 pyspark/sql/utils.py that dropped
#     it. The package directory has to be deleted outright.
#   * The runtime must then restart, because the old modules are already in
#     sys.modules.

import os
import shutil
import sysconfig

SENTINEL = "/content/.deequ_env_ready"

if os.path.exists(SENTINEL):
    print(f"✅ Environment already prepared. Delete {SENTINEL} to rebuild it.")
else:
    !apt-get update -qq
    !apt-get install -qq -y openjdk-11-jdk-headless > /dev/null

    !pip uninstall -y -q pyspark pydeequ dataproc-spark-connect pyspark-connect

    # pip leaves files behind often enough that this matters; one stale module
    # is all it takes to break the import.
    site_dir = sysconfig.get_paths()["purelib"]
    stale = os.path.join(site_dir, "pyspark")
    if os.path.isdir(stale):
        shutil.rmtree(stale)
        print(f"removed stale pyspark tree: {stale}")

    !pip install -q "pyspark==3.5.9" "pydeequ==1.6.0"

    with open(SENTINEL, "w") as fh:
        fh.write("ok")

    print("\n" + "=" * 68)
    print("RESTARTING THE RUNTIME so the new PySpark loads cleanly.")
    print("This cell is expected to interrupt 'Run all' exactly once.")
    print("When the runtime comes back, just run everything again — this cell")
    print("will detect the sentinel and skip straight through.")
    print("=" * 68)

    import IPython

    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Setup 2 — preflight: select a supported JDK and verify the coupled versions.
#    Spark surfaces nearly all of these as JAVA_GATEWAY_EXITED, which names no
#    cause, so resolve and check them explicitly and say what is actually wrong.
import glob
import os
import subprocess
import sys

problems = []

# --- Java -----------------------------------------------------------------
# Colab's default `java` is 21, which Spark 3.5 does not support, and apt
# installing openjdk-11 does NOT repoint /usr/bin/java. So pick the JDK
# directory directly instead of following the symlink.
jdks = (
    sorted(glob.glob("/usr/lib/jvm/java-11-openjdk*"))
    or sorted(glob.glob("/usr/lib/jvm/java-17-openjdk*"))
    or sorted(glob.glob("/usr/lib/jvm/java-8-openjdk*"))
)

if jdks:
    JAVA_HOME = jdks[0]
    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ["PATH"]
    banner = subprocess.run(
        [os.path.join(JAVA_HOME, "bin", "java"), "-version"],
        capture_output=True, text=True,
    ).stderr.splitlines()[0].strip()
    print(f"JAVA_HOME : {JAVA_HOME}")
    print(f"Java      : {banner}")
else:
    problems.append(
        "No supported JDK (11/17/8) found under /usr/lib/jvm. Re-run Setup 1."
    )

# --- Python ---------------------------------------------------------------
py = sys.version_info
print(f"Python    : {py.major}.{py.minor}.{py.micro}")
if (py.major, py.minor) > (3, 11):
    problems.append(
        f"PySpark 3.5.x declares Python 3.8–3.11; this runtime is "
        f"{py.major}.{py.minor}. This pipeline runs entirely in the JVM (no "
        f"Python UDFs), so it normally works anyway — but a serialization "
        f"error later points back here."
    )

# --- PySpark --------------------------------------------------------------
import pyspark

print(f"PySpark   : {pyspark.__version__}")
if not pyspark.__version__.startswith("3.5"):
    problems.append(
        f"PyDeequ supports Spark 3.5 only; found PySpark {pyspark.__version__}. "
        "Spark 4.x needs Java 17+ and is Scala 2.13, so it cannot load the "
        "Scala 2.12 Deequ JAR."
    )

# A mixed install fails much later with a confusing ImportError from inside
# pyspark.pandas, so detect it here. get_column_class exists in 3.5 and was
# removed in 4.0, which makes it a reliable marker of a half-downgraded tree.
try:
    from pyspark.sql.utils import get_column_class  # noqa: F401
except ImportError:
    problems.append(
        "Mixed PySpark installation: pyspark.sql.utils is missing "
        "get_column_class, which PySpark 3.5's pyspark.pandas imports. A 4.x "
        "tree was downgraded in place. Delete /content/.deequ_env_ready, then "
        "re-run Setup 1 and let it restart the runtime."
    )

if problems:
    print("\n⚠️  Environment warnings:")
    for p in problems:
        print(f"   - {p}")
else:
    print("\n✅ Version matrix is consistent.")

In [ ]:
# Setup 3 — Spark session.
import os
import sys

# JAVA_HOME was resolved and validated by the preflight above; fail loudly here
# rather than letting Spark report it as JAVA_GATEWAY_EXITED.
if "JAVA_HOME" not in os.environ:
    raise RuntimeError("JAVA_HOME is not set — run Setup 2 (preflight) first.")

# In local[*] mode the driver JVM is ALSO the executor, so this heap is the
# entire budget for the Deequ work. The default is 1g, which is not enough for
# ~13M rows and surfaces in Colab as a dead kernel ("your session crashed").
#
# This has to go through PYSPARK_SUBMIT_ARGS: SparkSession.builder.config() is
# applied after the JVM has already launched, so spark.driver.memory set there
# is silently ignored in local mode.
DRIVER_MEMORY = "6g"  # Colab standard runtime has ~12.7 GB; leave room for Python
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {DRIVER_MEMORY} pyspark-shell"

# PyDeequ reads SPARK_VERSION at import time to choose its Deequ JAR, so this
# must be set BEFORE `import pydeequ`. 3.5 is the only value it accepts.
os.environ["SPARK_VERSION"] = "3.5"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pydeequ
from pyspark.sql import SparkSession

# Take the coordinate from PyDeequ itself rather than hardcoding one, so the
# Python API and the JVM JAR cannot drift apart.
print(f"JAVA_HOME : {os.environ['JAVA_HOME']}")
print(f"Deequ JAR : {pydeequ.deequ_maven_coord}")

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")  # crucial for Colab stability
    .config("spark.jars.packages", pydeequ.deequ_maven_coord)
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    # 200 shuffle partitions (the default) is pure overhead on a single node.
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

# Confirm the Deequ classes actually resolved. py4j hands back a JavaPackage
# placeholder instead of raising when a class is missing, which is why the
# usual symptom is a confusing "JavaPackage object is not callable" much later.
_probe = spark._jvm.com.amazon.deequ.checks.Check
if _probe.__class__.__name__ == "JavaPackage":
    raise RuntimeError(
        "Deequ JAR did not load. The Maven download can fail silently; re-run "
        "this cell, and check the coordinate above matches the Spark version "
        "printed by the preflight."
    )

# Report the heap actually granted. If this says ~1 GB, a Spark session was
# already alive when this cell ran and getOrCreate() reused it — restart the
# runtime, because driver memory cannot be changed on a running JVM.
heap_gb = spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() / (1024 ** 3)
print(f"✅ Spark {spark.version} live, Deequ classes resolved")
print(f"   Driver heap: {heap_gb:.1f} GB (requested {DRIVER_MEMORY})")
if heap_gb < 2:
    print("   ⚠️  Heap is still at the default — restart the runtime and re-run.")

In [ ]:
# Download the three most recent published months from the NYC TLC endpoint.
# TLC publishes monthly with roughly a two-month lag — if a file comes back
# 0 bytes that month isn't published yet, so step the URLs back one month.
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-09.parquet" -O "sep_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-10.parquet" -O "oct_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet" -O "nov_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv" -O "zone_lookup.csv"

from pathlib import Path

empty = [f for f in ("sep_2025.parquet", "oct_2025.parquet", "nov_2025.parquet")
         if Path(f).exists() and Path(f).stat().st_size == 0]

if empty:
    for f in empty:
        Path(f).unlink()
    print(f"⚠️  Not published yet, removed: {empty}")
    print("   Step the URLs above back a month, or let the local-file fallback handle it.")
else:
    print("✅ All files downloaded")

!ls -lh *.parquet *.csv

In [ ]:
from pathlib import Path

# Resolve the project root so this notebook runs unchanged from either the repo
# root (Colab: /content) or from notebooks/ (local Jupyter).
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
print(f"📁 Project root: {PROJECT_ROOT}")

# Prefer 2025 monthly files if they exist (Colab / cloud run)
monthly_files_2025 = {
    "2025-09": "sep_2025.parquet",
    "2025-10": "oct_2025.parquet",
    "2025-11": "nov_2025.parquet",
}

# Fallback to the 2024 files if they are checked out locally
monthly_files_2024 = {
    "2024-01": "yellow_tripdata_2024-01.parquet",
    "2024-02": "yellow_tripdata_2024-02.parquet",
    "2024-03": "yellow_tripdata_2024-03.parquet",
}


def resolve(mapping):
    """Return {month: Path} for the files that exist, looking in the project
    root first and then the working directory."""
    found = {}
    for month, name in mapping.items():
        for base in (PROJECT_ROOT, CWD):
            candidate = base / name
            if candidate.exists():
                found[month] = candidate
                break
    return found


available_2025 = resolve(monthly_files_2025)

if available_2025:
    month_to_path = available_2025
    print("📂 Using 2025 NYC Taxi parquet files:")
else:
    available_2024 = resolve(monthly_files_2024)
    if not available_2024:
        raise FileNotFoundError(
            "No NYC Taxi parquet files found. "
            "Expected either the 2025 downloads (sep_2025, oct_2025, nov_2025) "
            f"or yellow_tripdata_2024-01..03.parquet under {PROJECT_ROOT}."
        )
    month_to_path = available_2024
    print("📂 Using local NYC Taxi parquet files:")

for month, path in sorted(month_to_path.items()):
    print(f"  - {month}: {path}")

# Build a month -> DataFrame mapping. The per-month frames are kept alongside
# the combined frame because the metrics store analyses each month separately.
monthly_dfs = {month: spark.read.parquet(str(path)) for month, path in month_to_path.items()}

ordered_months = sorted(monthly_dfs.keys())

# Labels derived from the data actually loaded, so every report header below
# stays truthful whichever months are present.
MONTH_LABEL = " / ".join(ordered_months)
MONTH_RANGE = f"{ordered_months[0]} to {ordered_months[-1]}"
DATASET_SLUG = f"{ordered_months[0]}_{ordered_months[-1]}".replace("-", "")

# allowMissingColumns keeps the union working across a schema change — 2025
# files carry cbd_congestion_fee, earlier years do not.
df = None
for month in ordered_months:
    sdf = monthly_dfs[month]
    df = sdf if df is None else df.unionByName(sdf, allowMissingColumns=True)

# Row counts per month and overall
print("\n✅ Row counts by month:")
for month in ordered_months:
    cnt = monthly_dfs[month].count()
    print(f"  - {month}: {cnt:,} rows")

print(f"\n✅ Total combined rows: {df.count():,}")
print(f"✅ Columns: {len(df.columns)}")
print()
df.printSchema()
df.show(5, truncate=False)

In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult

check = Check(spark, CheckLevel.Warning, "NYC Taxi Quality Checks")

checkResult = (VerificationSuite(spark)
    .onData(df)
    .addCheck(
        check
        # Completeness checks
        .isComplete("tpep_pickup_datetime")
        .isComplete("tpep_dropoff_datetime")
        .isComplete("fare_amount")
        # Value validity checks
        .isNonNegative("fare_amount")
        .isNonNegative("tip_amount")
        .isNonNegative("trip_distance")
        # Business rule checks
        .satisfies("trip_distance > 0", "Trip distance must be positive")
        .satisfies("fare_amount > 0", "Revenue integrity: fare must be positive")
        .satisfies("tpep_dropoff_datetime > tpep_pickup_datetime", "Dropoff must be after pickup")
        .satisfies("passenger_count >= 1 AND passenger_count <= 6", "Passenger count must be 1-6")
        .satisfies("PULocationID >= 1 AND PULocationID <= 265", "Pickup zone must be valid NYC zone")
        .satisfies("DOLocationID >= 1 AND DOLocationID <= 265", "Dropoff zone must be valid NYC zone")
    ).run()
)

print("✅ Checks complete")

In [ ]:
results_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
results_df.select("constraint", "constraint_status", "constraint_message").show(20, truncate=False)

In [ ]:
import pandas as pd

pdf = results_df.toPandas()
total = len(pdf)
passed = len(pdf[pdf["constraint_status"] == "Success"])
failed = total - passed

print("=" * 55)
print(f"  NYC Taxi Data Quality Report — {MONTH_LABEL}")
print("=" * 55)
print(f"  Total checks : {total}")
print(f"  ✅ Passed    : {passed}")
print(f"  🚨 Failed    : {failed}")
print("=" * 55)
if failed > 0:
    print("\n  Failed checks:")
    for _, row in pdf[pdf["constraint_status"] != "Success"].iterrows():
        print(f"  → {row['constraint']}")

In [ ]:
from pyspark.sql.functions import col, count, lit, when
from pyspark.sql.functions import sum as spark_sum

print("📊 Quantifying data quality issues...\n")

# Each predicate mirrors the Deequ constraint it corresponds to, so the row
# counts here and the compliance ratios above describe the same thing.
issue_predicates = {
    "Negative fare_amount": col("fare_amount") < 0,
    "Zero or negative fare": col("fare_amount") <= 0,
    "Negative tip_amount": col("tip_amount") < 0,
    "Zero trip distance": col("trip_distance") <= 0,
    # NULL is counted on purpose: Deequ's satisfies() treats a NULL predicate as
    # a violation, so excluding nulls would make this row disagree with the
    # constraint compliance ratio reported above.
    "Invalid passenger count": (
        col("passenger_count").isNull()
        | (col("passenger_count") < 1)
        | (col("passenger_count") > 6)
    ),
    "Dropoff before pickup": col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"),
}

# One pass. Counting these separately meant seven full scans of the union —
# re-reading all three parquet files each time, for identical numbers.
aggregates = [count(lit(1)).alias("total_rows")] + [
    spark_sum(when(predicate, 1).otherwise(0)).alias(f"issue_{i}")
    for i, predicate in enumerate(issue_predicates.values())
]

row = df.agg(*aggregates).collect()[0]

total_rows = int(row["total_rows"])
issues = {name: int(row[f"issue_{i}"]) for i, name in enumerate(issue_predicates)}

print(f"Total rows analyzed: {total_rows:,}\n")
print(f"{'Issue':<35} {'Rows Affected':>15} {'% of Data':>10}")
print("-" * 63)
for issue, count_val in issues.items():
    pct = (count_val / total_rows) * 100
    print(f"{issue:<35} {count_val:>15,} {pct:>9.2f}%")

In [ ]:
# 3. Column Profiling — automated statistics for every column
from pydeequ.profiles import ColumnProfilerRunner

print("📊 Running Deequ column profiler across all columns...\n")

profile_result = (
    ColumnProfilerRunner(spark)
    .onData(df)
    .run()
)

# Convert the profiler output into a Spark + pandas-friendly summary
profile_rows = []
for col_name, col_profile in profile_result.profiles.items():
    summary = {
        "column": col_name,
        "dataType": str(col_profile.dataType()),
        "completeness": float(col_profile.completeness()),
        "approxDistinctCount": int(col_profile.approximateNumDistinctValues()),
    }

    if hasattr(col_profile, "mean") and col_profile.mean() is not None:
        summary["mean"] = float(col_profile.mean())
    if hasattr(col_profile, "stdDev") and col_profile.stdDev() is not None:
        summary["stddev"] = float(col_profile.stdDev())
    if hasattr(col_profile, "maximum") and col_profile.maximum() is not None:
        summary["max"] = float(col_profile.maximum()) if isinstance(col_profile.maximum(), (int, float)) else col_profile.maximum()
    if hasattr(col_profile, "minimum") and col_profile.minimum() is not None:
        summary["min"] = float(col_profile.minimum()) if isinstance(col_profile.minimum(), (int, float)) else col_profile.minimum()

    profile_rows.append(summary)

profile_df = spark.createDataFrame(profile_rows)

print("✅ Column profiling complete. Top-level summary (sorted by incompleteness):\n")
(profile_df
    .orderBy(profile_df.completeness.asc())
    .show(30, truncate=False))

# Keep a pandas copy handy for reporting / EDA
profile_pdf = profile_df.toPandas()

In [ ]:
# 4. Metrics repository — persist Deequ metrics so months can be compared over
#    time. This is the piece that turns one-off checks into drift detection.
from pydeequ.repository import FileSystemMetricsRepository, ResultKey
from pydeequ.analyzers import (
    AnalysisRunner,
    Size,
    Mean,
    StandardDeviation,
    Completeness,
    Maximum,
    Minimum,
    CountDistinct,
)

metrics_dir = PROJECT_ROOT / "results" / "metrics"
reports_dir = PROJECT_ROOT / "results" / "reports"
metrics_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

metrics_path = metrics_dir / "taxi_metrics.json"

# Swap this single line for an s3:// URI to persist metrics permanently in
# production — see docs/aws-deployment-guide.md.
repository = FileSystemMetricsRepository(spark, str(metrics_path))

print(f"✅ Metrics repository ready at: {metrics_path}")

In [ ]:
# 5. Run analysis on each month and persist metrics
for month, month_df in monthly_dfs.items():
    print(f"⏳ Analyzing {month}...")

    resultKey = ResultKey(
        spark,
        ResultKey.current_milli_time(),
        {"month": month, "dataset": "nyc_yellow_taxi"},
    )

    (
        AnalysisRunner(spark)
        .onData(month_df)
        .addAnalyzer(Size())
        .addAnalyzer(Mean("fare_amount"))
        .addAnalyzer(Mean("trip_distance"))
        .addAnalyzer(Mean("passenger_count"))
        .addAnalyzer(StandardDeviation("fare_amount"))
        .addAnalyzer(Maximum("fare_amount"))
        .addAnalyzer(Minimum("fare_amount"))
        .addAnalyzer(Completeness("passenger_count"))
        .addAnalyzer(Completeness("fare_amount"))
        .addAnalyzer(CountDistinct("PULocationID"))
        .useRepository(repository)
        .saveOrAppendResult(resultKey)
        .run()
    )

    print(f"✅ {month} saved to metrics store")

print("\n✅ All months analyzed and stored")

In [ ]:
# 6. Load the persisted metrics back out of the repository. Every run appends,
#    so this frame grows into a real time series across months.
metrics_df = (
    repository
    .load()
    .before(ResultKey.current_milli_time())
    .getSuccessMetricsAsDataFrame()
)

print(f"✅ Loaded {metrics_df.count()} metric rows from the repository\n")
metrics_df.show(50, truncate=False)

In [ ]:
# 7. Visualize metric trends across months (drift / anomalies)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

# Ensure we have a stable month ordering matching our data load
months_label = sorted(monthly_dfs.keys())

metrics_pdf = metrics_df.toPandas()

# Extract the month tag from the repository tags column
if "tags" in metrics_pdf.columns:
    def extract_month(tags):
        if isinstance(tags, dict):
            return tags.get("month")
        return None

    metrics_pdf["month"] = metrics_pdf["tags"].apply(extract_month)
else:
    # Fallback: no tags column, use months_label in insertion order
    metrics_pdf["month"] = None

# Helper to pull a metric's values in month order
def get_metric(name, instance):
    subset = metrics_pdf[
        (metrics_pdf["name"] == name)
        & (metrics_pdf["instance"] == instance)
    ][["month", "value"]].copy()

    # Align to our known month order
    subset = subset.set_index("month").reindex(months_label)
    subset = subset.reset_index().rename(columns={"index": "month"})
    return subset

mean_fare    = get_metric("Mean", "fare_amount")
mean_dist    = get_metric("Mean", "trip_distance")
row_count    = get_metric("Size", "*")
completeness = get_metric("Completeness", "passenger_count")

fig = plt.figure(figsize=(14, 10))
fig.suptitle(
    "NYC Yellow Taxi — Data Quality Metrics Over Time",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# Plot 1 — Mean Fare
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(months_label, mean_fare["value"].values, marker="o", color="#e74c3c", linewidth=2)
ax1.set_title("Mean Fare Amount ($)")
ax1.set_ylabel("USD")
ax1.grid(True, alpha=0.3)

# Plot 2 — Mean Trip Distance
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(months_label, mean_dist["value"].values, marker="o", color="#3498db", linewidth=2)
ax2.set_title("Mean Trip Distance (miles)")
ax2.set_ylabel("Miles")
ax2.grid(True, alpha=0.3)

# Plot 3 — Row Count
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(months_label, row_count["value"].values, color="#2ecc71", alpha=0.8)
ax3.set_title("Total Trips Per Month")
ax3.set_ylabel("Row Count")
ax3.grid(True, alpha=0.3, axis="y")

# Plot 4 — Passenger Count Completeness
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(
    months_label,
    completeness["value"].values,
    marker="s",
    color="#f39c12",
    linewidth=2,
)
ax4.set_title("Passenger Count Completeness")
ax4.set_ylabel("Completeness (0–1)")
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

output_chart_path = os.path.join(metrics_dir, "anomaly_trends.png")
plt.savefig(output_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Chart saved to {output_chart_path}")

In [ ]:
# 8. Consolidated human-readable quality + drift report

print("=" * 70)
print("NYC Taxi Data Quality & Drift Summary")
print("=" * 70)

# Constraint check summary from earlier cells
pdf_checks = results_df.toPandas()
num_checks = len(pdf_checks)
num_passed = (pdf_checks["constraint_status"] == "Success").sum()
num_failed = num_checks - num_passed

print(f"Total constraints evaluated : {num_checks}")
print(f"✅ Constraints passed       : {num_passed}")
print(f"🚨 Constraints failed       : {num_failed}")

if num_failed > 0:
    print("\nFailed constraints:")
    for _, row in pdf_checks[pdf_checks["constraint_status"] != "Success"].iterrows():
        print(f"  → {row['constraint']}")

print("\nKey issue counts (from full-data scan):")

# Issue counts from earlier cell
issue_counts = issues

for name, count_val in issue_counts.items():
    pct = (count_val / total_rows) * 100 if total_rows else 0.0
    print(f"  - {name:<30} {count_val:>10,} rows  ({pct:>6.2f}% of data)")

# Simple drift narrative based on metrics
metrics_pdf = metrics_df.toPandas().copy()

if "tags" in metrics_pdf.columns:
    def extract_month(tags):
        if isinstance(tags, dict):
            return tags.get("month")
        return None
    metrics_pdf["month"] = metrics_pdf["tags"].apply(extract_month)

months_label = sorted(monthly_dfs.keys())

def metric_series(name, instance):
    subset = metrics_pdf[
        (metrics_pdf["name"] == name)
        & (metrics_pdf["instance"] == instance)
    ][["month", "value"]]
    subset = subset.set_index("month").reindex(months_label)
    return subset["value"]

try:
    mean_fare_series = metric_series("Mean", "fare_amount")
    first_fare = float(mean_fare_series.iloc[0])
    last_fare = float(mean_fare_series.iloc[-1])
    delta_fare = last_fare - first_fare
    pct_change_fare = (delta_fare / first_fare) * 100 if first_fare else 0.0

    print("\nDrift highlights:")
    print(f"  - Mean fare changed from ${first_fare:0.2f} to ${last_fare:0.2f} across {len(months_label)} months ({pct_change_fare:+0.2f}% change)")

    mean_dist_series = metric_series("Mean", "trip_distance")
    first_dist = float(mean_dist_series.iloc[0])
    last_dist = float(mean_dist_series.iloc[-1])
    delta_dist = last_dist - first_dist
    pct_change_dist = (delta_dist / first_dist) * 100 if first_dist else 0.0

    print(f"  - Mean trip distance changed from {first_dist:0.2f} to {last_dist:0.2f} miles ({pct_change_dist:+0.2f}% change)")

    compl_series = metric_series("Completeness", "passenger_count")
    first_compl = float(compl_series.iloc[0])
    last_compl = float(compl_series.iloc[-1])
    delta_compl = last_compl - first_compl

    print(f"  - Passenger count completeness moved from {first_compl:0.4f} to {last_compl:0.4f} ({delta_compl:+0.4f} absolute change)")
except Exception as e:
    print("\n(Drift narrative could not be fully computed — check metrics repository content.)")
    print(f"Details: {e}")

print("\n✅ Consolidated quality and drift report ready for README / LinkedIn narrative.")

In [ ]:
# 9. Export machine-readable artefacts + a paste-ready README block.
import pandas as pd

constraint_csv = reports_dir / f"constraint_results_{DATASET_SLUG}.csv"
pdf_checks.to_csv(constraint_csv, index=False)

profile_csv = reports_dir / f"column_profile_{DATASET_SLUG}.csv"
profile_pdf.to_csv(profile_csv, index=False)

issues_pdf = pd.DataFrame(
    [
        {
            "issue": name,
            "rows_affected": int(count_val),
            "pct_of_data": round((count_val / total_rows) * 100, 4),
        }
        for name, count_val in issues.items()
    ]
)
issues_csv = reports_dir / f"issue_counts_{DATASET_SLUG}.csv"
issues_pdf.to_csv(issues_csv, index=False)

metrics_csv = reports_dir / f"metrics_{DATASET_SLUG}.csv"
metrics_pdf.to_csv(metrics_csv, index=False)


def fmt(name, spec):
    """Format a drift value by name, tolerating the metrics repository having
    been only partially populated."""
    try:
        return format(float(globals().get(name)), spec)
    except (TypeError, ValueError):
        return "n/a"


issue_label = {
    "Invalid passenger count": "Invalid passenger count (outside 1–6)",
    "Zero or negative fare": "Zero or negative fare amount",
    "Negative fare_amount": "Negative fare amount",
    "Zero trip distance": "Zero trip distance",
    "Dropoff before pickup": "Dropoff recorded before pickup",
    "Negative tip_amount": "Negative tip amount",
}
issue_order = [
    "Invalid passenger count",
    "Zero or negative fare",
    "Negative fare_amount",
    "Zero trip distance",
    "Dropoff before pickup",
    "Negative tip_amount",
]

lines = [
    "## 📊 Results",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| Months analyzed | {', '.join(ordered_months)} |",
    f"| Total rows validated | {total_rows:,} |",
    f"| Constraint checks run | {num_checks} |",
    f"| ✅ Checks passed | {num_passed} |",
    f"| 🚨 Checks failed | {num_failed} |",
    "",
    "### Real Data Quality Issues Found",
    "",
    "| Issue | Rows Affected | % of Data |",
    "|---|---|---|",
]

for key in issue_order:
    if key not in issues:
        continue
    rows_affected = issues[key]
    pct = (rows_affected / total_rows) * 100 if total_rows else 0.0
    lines.append(f"| {issue_label[key]} | {rows_affected:,} | {pct:.2f}% |")

lines += [
    "",
    f"### Drift Detected Across {len(ordered_months)} Months",
    f"- Mean fare amount: ${fmt('first_fare', '.2f')} → ${fmt('last_fare', '.2f')} "
    f"({fmt('pct_change_fare', '+.2f')}% change)",
    f"- Mean trip distance: {fmt('first_dist', '.2f')} → {fmt('last_dist', '.2f')} miles "
    f"({fmt('pct_change_dist', '+.2f')}% change)",
    f"- Passenger count completeness: {fmt('first_compl', '.4f')} → "
    f"{fmt('last_compl', '.4f')}",
    "",
]

snippet_path = reports_dir / "README_results_snippet.md"
snippet_path.write_text("\n".join(lines), encoding="utf-8")

print("✅ Exported:")
for p in (constraint_csv, profile_csv, issues_csv, metrics_csv, snippet_path):
    print(f"  - {p.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 70)
print("Paste the block below straight into README.md")
print("=" * 70)
print("\n".join(lines))